# DroneGuard: train & evaluate on VisDrone (Colab)

Runtime → Change runtime type → **GPU** (T4 is enough for the baseline; L4/A100 for 1024px).

Pipeline: download once → convert with fixed labels → baseline (YOLOv8n, 640) → upgraded model (YOLO11s, 1024) → per-class evaluation on the **val** split → demo with **best.pt**.

In [ ]:
# Get the code (replace with your GitHub repo URL once pushed)
REPO_URL = "https://github.com/<your-username>/droneguard.git"
!git clone -q $REPO_URL droneguard || echo 'Clone failed: upload the droneguard folder manually'
%cd droneguard
!pip install -q -r requirements.txt

In [ ]:
# Run the converter tests first: catches label-mapping regressions in seconds
!python -m pytest -q tests

In [ ]:
# 1) Download VisDrone once (cached by kagglehub)
import kagglehub
SRC = kagglehub.dataset_download("kushagrapandya/visdrone-dataset")
print(SRC)
!ls $SRC

In [ ]:
# 2) Convert to YOLO format (skips ignored regions, 'others', and score-0 boxes; uses the real val split)
!python -m droneguard prepare --src "$SRC" --dst datasets/visdrone-yolo
!cat datasets/visdrone-yolo/data.yaml
!cat datasets/visdrone-yolo/conversion_stats.json

## 3) Baseline: same model as the original project, but with correct labels
This gives an honest before/after: the original run's numbers were invalid (shifted labels, validated on train).

In [ ]:
!python -m droneguard train --model yolov8n.pt --imgsz 640 --epochs 50 --name baseline_v8n_640
!python -m droneguard evaluate --weights runs/detect/baseline_v8n_640/weights/best.pt --imgsz 640 --out results/baseline_v8n_640.csv

## 4) Upgraded model: newer architecture + higher resolution
VisDrone objects are tiny, so resolution matters more than model size. Drop `--imgsz` to 960 or `--model yolo11n.pt` if you run out of GPU memory.

In [ ]:
!python -m droneguard train --model yolo11s.pt --imgsz 1024 --epochs 100 --name yolo11s_1024
!python -m droneguard evaluate --weights runs/detect/yolo11s_1024/weights/best.pt --imgsz 1024 --out results/yolo11s_1024.csv

In [ ]:
# 5) Compare runs
import pandas as pd
b = pd.read_csv('results/baseline_v8n_640.csv').set_index('class')
u = pd.read_csv('results/yolo11s_1024.csv').set_index('class')
pd.concat({'baseline mAP50-95': b['mAP50-95'], 'upgraded mAP50-95': u['mAP50-95'], 'upgraded mAP50': u['mAP50']}, axis=1).round(3)

In [ ]:
# 6) Demo with YOUR trained weights (not a generic COCO checkpoint)
!python -m droneguard predict --weights runs/detect/yolo11s_1024/weights/best.pt --source datasets/visdrone-yolo/images/val --imgsz 1024 --name val_demo
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('runs/predict/val_demo/*.jpg'))[:3]:
    display(Image(p, width=900))

In [ ]:
# 7) Save results to Drive so they survive the Colab session
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/droneguard
!cp -r results runs/detect/*/weights runs/detect/*/results.png /content/drive/MyDrive/droneguard/ 2>/dev/null; ls /content/drive/MyDrive/droneguard